In [ ]:
# BLOCK 1 — Data Loading & Preprocessing (no oversampling copy)
import warnings; warnings.filterwarnings('ignore')
import torch, numpy as np, pandas as pd, pickle, json, os, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

os.makedirs('paper_results', exist_ok=True)

CSV_PATH = "/kaggle/input/datasets/ashhh1011/nf-ton/NF-ToN-IoT-v2.csv"
df = pd.read_csv(CSV_PATH)

le = LabelEncoder()
y  = le.fit_transform(df['Attack'].values)
num_classes = len(le.classes_)
class_names = list(le.classes_)

drop_cols = [c for c in ['IPV4_SRC_ADDR','IPV4_DST_ADDR','Attack','Label'] if c in df.columns]
X = df.drop(columns=drop_cols).values.astype(np.float64)

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
scaler = StandardScaler()
X = scaler.fit_transform(X)
X = np.clip(np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0), -10, 10).astype(np.float32)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_search, y_train, y_search = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

train_counts = np.array([(y_train==i).sum() for i in range(num_classes)], dtype=np.float64)
cbrt_w = np.cbrt(train_counts.max() / train_counts)
cbrt_w = cbrt_w / cbrt_w.mean()

block1_data = {
    'X_train':X_train, 'y_train':y_train,
    'X_search':X_search, 'y_search':y_search,
    'X_retrain':np.vstack([X_train,X_search]),
    'y_retrain':np.concatenate([y_train,y_search]),
    'X_test':X_test, 'y_test':y_test,
    'input_dim':X.shape[1], 'num_classes':num_classes,
    'class_names':class_names, 'train_counts':train_counts,
}
with open('paper_results/block1_data.pkl','wb') as f: pickle.dump(block1_data, f)
with open('paper_results/label_encoder.pkl','wb') as f: pickle.dump(le, f)

print("✅ Block 1 complete (memory‑safe)")

In [ ]:
# BLOCK 2  DARTS Architecture Search (with full metrics logging)
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim, copy, pickle, json, numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, matthews_corrcoef, roc_auc_score

with open('paper_results/block1_data.pkl','rb') as f: data = pickle.load(f)

device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes  = data['num_classes']
input_dim    = data['input_dim']
class_names  = data['class_names']
train_counts = data['train_counts']

X_train_gpu  = torch.FloatTensor(data['X_train']).to(device)
y_train_gpu  = torch.LongTensor(data['y_train']).to(device)
X_search_gpu = torch.FloatTensor(data['X_search']).to(device)
y_search_gpu = torch.LongTensor(data['y_search']).to(device)
X_test_gpu   = torch.FloatTensor(data['X_test']).to(device)
y_test       = data['y_test']

cbrt_w    = np.cbrt(train_counts.max() / train_counts); cbrt_w /= cbrt_w.mean()
cw_tensor = torch.FloatTensor(cbrt_w).to(device)

# ── Ops ─────────────────────────────
class LinearReLU(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.ReLU())
    def forward(self,x): return self.op(x)

class Residual(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.ReLU())
    def forward(self,x): return x + self.op(x)

class SEBlock(nn.Module):
    def __init__(self,dim):
        super().__init__(); r=max(4,dim//8)
        self.se=nn.Sequential(nn.Linear(dim,r),nn.ReLU(),nn.Linear(r,dim),nn.Sigmoid())
        self.ln=nn.Linear(dim,dim)
    def forward(self,x): return self.ln(x)*self.se(x)

class GatedLinear(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.fc=nn.Linear(dim,dim*2); self.bn=nn.BatchNorm1d(dim)
    def forward(self,x):
        h=self.fc(x); v,g=h.chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))

class Bottleneck(nn.Module):
    def __init__(self,dim):
        super().__init__(); mid=max(8,dim//4)
        self.op=nn.Sequential(nn.Linear(dim,mid),nn.ReLU(),nn.Linear(mid,dim),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS = {
    'linear_relu':LinearReLU,
    'bottleneck':Bottleneck,
    'residual':Residual,
    'se_block':SEBlock,
    'gated_linear':GatedLinear,
}
OP_NAMES = list(OPS.keys())

# ── Supernet ────────────────────────
class MixedOp(nn.Module):
    def __init__(self,dim): super().__init__(); self.ops=nn.ModuleList([OPS[n](dim) for n in OP_NAMES])
    def forward(self,x,w): return sum(wi*op(x) for wi,op in zip(w,self.ops))

class DARTSCell(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.mixed=MixedOp(dim); self.alphas=nn.Parameter(torch.zeros(len(OP_NAMES)))
    def forward(self,x): return self.mixed(x, F.softmax(self.alphas,dim=0))
    def get_best(self):
        w=F.softmax(self.alphas,dim=0).detach().cpu().numpy()
        idx=np.argmax(w); return OP_NAMES[idx],float(w[idx]),w

class DARTSSupernet(nn.Module):
    def __init__(self,input_dim,hidden=384,num_classes=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(input_dim,hidden),nn.BatchNorm1d(hidden),nn.ReLU(),nn.Dropout(0.05))
        self.cells=nn.ModuleList([DARTSCell(hidden) for _ in range(4)])
        self.head=nn.Sequential(nn.Dropout(0.15),nn.Linear(hidden,num_classes))
    def forward(self,x):
        x=self.proj(x)
        for c in self.cells: x=c(x)
        return self.head(x)
    def arch_params(self): return [c.alphas for c in self.cells]
    def weight_params(self):
        aids={id(p) for p in self.arch_params()}
        return [p for p in self.parameters() if id(p) not in aids]

HIDDEN=384
supernet = DARTSSupernet(input_dim, HIDDEN, num_classes).to(device)

w_opt = optim.AdamW(supernet.weight_params(), lr=1e-3, weight_decay=2e-4)
a_opt = optim.Adam(supernet.arch_params(),    lr=3e-4, betas=(0.5,0.999))
crit  = nn.CrossEntropyLoss(weight=cw_tensor, label_smoothing=0.05)

SEARCH_EPOCHS=60; SEARCH_BATCH=2048; PATIENCE=15
best_f1=0.0; patience=0

def eval_metrics(model,X_gpu,y_true,batch=4096):
    model.eval(); preds=[]; probs=[]
    with torch.no_grad():
        for i in range(0,len(X_gpu),batch):
            logits=model(X_gpu[i:i+batch])
            preds.extend(logits.argmax(1).cpu().numpy())
            probs.extend(torch.softmax(logits,dim=1).cpu().numpy())
    preds=np.array(preds); probs=np.array(probs)
    f1 = f1_score(y_true,preds,average='macro',zero_division=0)
    acc= accuracy_score(y_true,preds)
    prec=precision_score(y_true,preds,average='macro',zero_division=0)
    rec =recall_score(y_true,preds,average='macro',zero_division=0)
    mcc =matthews_corrcoef(y_true,preds)
    try:
        auc =roc_auc_score(y_true,probs,multi_class='ovr',average='macro')
    except:
        auc =0.0
    return f1,acc,prec,rec,mcc,auc

print(f"Search start | epochs={SEARCH_EPOCHS} | batch={SEARCH_BATCH}")

for epoch in range(SEARCH_EPOCHS):
    supernet.train()
    idx = torch.randperm(len(X_train_gpu), device=device)
    for i in range(0, len(X_train_gpu), SEARCH_BATCH):
        xb=X_train_gpu[idx[i:i+SEARCH_BATCH]]; yb=y_train_gpu[idx[i:i+SEARCH_BATCH]]
        w_opt.zero_grad(); crit(supernet(xb),yb).backward()
        nn.utils.clip_grad_norm_(supernet.weight_params(),1.0); w_opt.step()

    idx2 = torch.randperm(len(X_search_gpu), device=device)
    for i in range(0, len(X_search_gpu), SEARCH_BATCH):
        xb=X_search_gpu[idx2[i:i+SEARCH_BATCH]]; yb=y_search_gpu[idx2[i:i+SEARCH_BATCH]]
        a_opt.zero_grad(); crit(supernet(xb),yb).backward(); a_opt.step()

    f1,acc,prec,rec,mcc,auc = eval_metrics(supernet, X_test_gpu, y_test)

    if f1 > best_f1:
        best_f1=f1; patience=0
        torch.save(supernet.state_dict(),'paper_results/best_supernet.pth')
        print(f"💾 Ep{epoch+1:03d} | F1:{f1:.4f} Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} MCC:{mcc:.4f} AUC:{auc:.4f}")
    else:
        patience+=1
        if (epoch+1)%5==0:
            print(f"   Ep{epoch+1:03d} | F1:{f1:.4f} Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} MCC:{mcc:.4f} AUC:{auc:.4f} | Pat:{patience}")
        if patience>=PATIENCE: break

supernet.load_state_dict(torch.load('paper_results/best_supernet.pth'))
best_arch={}
for i,c in enumerate(supernet.cells,1):
    op,w,_=c.get_best()
    best_arch[f'cell{i}']={'best_op':op,'weight':w}

with open('paper_results/best_architecture.json','w') as f: json.dump(best_arch,f,indent=2)
print("✅ Block 2 complete")

In [ ]:

# BLOCK 2 — DARTS SEARCH (FULL + CURVES + METRICS PER EPOCH)

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim, pickle, json, numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, matthews_corrcoef, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns

with open('paper_results/block1_data.pkl','rb') as f: data = pickle.load(f)

device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes  = data['num_classes']
input_dim    = data['input_dim']
train_counts = data['train_counts']

X_train_gpu  = torch.FloatTensor(data['X_train']).to(device)
y_train_gpu  = torch.LongTensor(data['y_train']).to(device)
X_search_gpu = torch.FloatTensor(data['X_search']).to(device)
y_search_gpu = torch.LongTensor(data['y_search']).to(device)

X_test = data['X_test']; y_test = data['y_test']

cbrt_w    = np.cbrt(train_counts.max() / train_counts); cbrt_w /= cbrt_w.mean()
cw_tensor = torch.FloatTensor(cbrt_w).to(device)

class LinearReLU(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.ReLU())
    def forward(self,x): return self.op(x)
class Residual(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.ReLU())
    def forward(self,x): return x + self.op(x)
class SEBlock(nn.Module):
    def __init__(self,dim):
        super().__init__(); r=max(4,dim//8)
        self.se=nn.Sequential(nn.Linear(dim,r),nn.ReLU(),nn.Linear(r,dim),nn.Sigmoid())
        self.ln=nn.Linear(dim,dim)
    def forward(self,x): return self.ln(x)*self.se(x)
class GatedLinear(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.fc=nn.Linear(dim,dim*2); self.bn=nn.BatchNorm1d(dim)
    def forward(self,x):
        h=self.fc(x); v,g=h.chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))
class Bottleneck(nn.Module):
    def __init__(self,dim):
        super().__init__(); mid=max(8,dim//4)
        self.op=nn.Sequential(nn.Linear(dim,mid),nn.ReLU(),nn.Linear(mid,dim),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS = {'linear_relu':LinearReLU,'bottleneck':Bottleneck,'residual':Residual,'se_block':SEBlock,'gated_linear':GatedLinear}
OP_NAMES = list(OPS.keys())

class MixedOp(nn.Module):
    def __init__(self,dim): super().__init__(); self.ops=nn.ModuleList([OPS[n](dim) for n in OP_NAMES])
    def forward(self,x,w): return sum(wi*op(x) for wi,op in zip(w,self.ops))

class DARTSCell(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.mixed=MixedOp(dim); self.alphas=nn.Parameter(torch.zeros(len(OP_NAMES)))
    def forward(self,x): return self.mixed(x, F.softmax(self.alphas,dim=0))
    def get_best(self):
        w=F.softmax(self.alphas,dim=0).detach().cpu().numpy()
        idx=np.argmax(w); return OP_NAMES[idx],float(w[idx]),w

class DARTSSupernet(nn.Module):
    def __init__(self,input_dim,hidden=512,num_classes=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(input_dim,hidden),nn.BatchNorm1d(hidden),nn.ReLU(),nn.Dropout(0.05))
        self.cells=nn.ModuleList([DARTSCell(hidden) for _ in range(4)])
        self.head=nn.Sequential(nn.Dropout(0.15),nn.Linear(hidden,num_classes))
    def forward(self,x):
        x=self.proj(x)
        for c in self.cells: x=c(x)
        return self.head(x)
    def arch_params(self): return [c.alphas for c in self.cells]
    def weight_params(self):
        aids={id(p) for p in self.arch_params()}
        return [p for p in self.parameters() if id(p) not in aids]

supernet = DARTSSupernet(input_dim, 512, num_classes).to(device)

w_opt = optim.AdamW(supernet.weight_params(), lr=1e-3, weight_decay=2e-4)
a_opt = optim.Adam(supernet.arch_params(),    lr=3e-4, betas=(0.5,0.999))
crit  = nn.CrossEntropyLoss(weight=cw_tensor, label_smoothing=0.05)

SEARCH_EPOCHS=50; SEARCH_BATCH=1024; PATIENCE=10
best_f1=0.0; patience=0
train_losses=[]; val_f1s=[]; val_accs=[]

for epoch in range(SEARCH_EPOCHS):
    torch.set_grad_enabled(True)
    supernet.train()
    for p in supernet.parameters(): p.requires_grad_(True)

    idx = torch.randperm(len(X_train_gpu), device=device)
    epoch_loss = 0.0; steps=0
    for i in range(0, len(X_train_gpu), SEARCH_BATCH):
        xb=X_train_gpu[idx[i:i+SEARCH_BATCH]]; yb=y_train_gpu[idx[i:i+SEARCH_BATCH]]
        w_opt.zero_grad()
        loss = crit(supernet(xb),yb)
        loss.backward()
        nn.utils.clip_grad_norm_(supernet.weight_params(),1.0)
        w_opt.step()
        epoch_loss += loss.item(); steps += 1

    idx2 = torch.randperm(len(X_search_gpu), device=device)
    for i in range(0, len(X_search_gpu), SEARCH_BATCH):
        xb=X_search_gpu[idx2[i:i+SEARCH_BATCH]]; yb=y_search_gpu[idx2[i:i+SEARCH_BATCH]]
        a_opt.zero_grad(); crit(supernet(xb),yb).backward(); a_opt.step()

    # ==== eval each epoch with full metrics ====
    supernet.eval(); preds=[]; probs=[]
    with torch.no_grad():
        for i in range(0,len(X_test),2048):
            xb=torch.FloatTensor(X_test[i:i+2048]).to(device)
            logits=supernet(xb)
            preds.extend(logits.argmax(1).cpu().numpy())
            probs.extend(torch.softmax(logits,dim=1).cpu().numpy())

    preds=np.array(preds); probs=np.array(probs)
    f1  = f1_score(y_test,preds,average='macro',zero_division=0)
    acc = accuracy_score(y_test,preds)
    prec= precision_score(y_test,preds,average='macro',zero_division=0)
    rec = recall_score(y_test,preds,average='macro',zero_division=0)
    mcc = matthews_corrcoef(y_test,preds)
    try: auc = roc_auc_score(y_test,probs,multi_class='ovr',average='macro')
    except: auc = 0.0

    train_losses.append(epoch_loss/steps); val_f1s.append(f1); val_accs.append(acc)

    print(f"Ep{epoch+1:03d} | F1:{f1:.4f} Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} MCC:{mcc:.4f} AUC:{auc:.4f}")

    if f1 > best_f1:
        best_f1=f1; patience=0
        torch.save(supernet.state_dict(),'paper_results/best_supernet.pth')
    else:
        patience+=1
        if patience>=PATIENCE: break

supernet.load_state_dict(torch.load('paper_results/best_supernet.pth'))
best_arch={}
for i,c in enumerate(supernet.cells,1):
    op,w,_=c.get_best()
    best_arch[f'cell{i}']={'best_op':op,'weight':w}
with open('paper_results/best_architecture.json','w') as f: json.dump(best_arch,f,indent=2)
print("✅ Block 2 complete")

# plots after training (same as before)

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import pickle, json, numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, matthews_corrcoef, roc_auc_score

with open('paper_results/block1_data.pkl','rb') as f: data=pickle.load(f)
with open('paper_results/best_architecture.json') as f: best_arch=json.load(f)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes=data['num_classes']; input_dim=data['input_dim']; train_counts=data['train_counts']
X_retrain,y_retrain = data['X_retrain'],data['y_retrain']
X_test,y_test       = data['X_test'],data['y_test']

cw = np.sqrt(train_counts.max()/train_counts); cw = cw/cw.mean()
cw_tensor=torch.FloatTensor(cw).to(device)

class Residual(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.BatchNorm1d(d),nn.ReLU())
    def forward(self,x): return x+self.op(x)
class SEBlock(nn.Module):
    def __init__(self,d):
        super().__init__(); r=max(4,d//8)
        self.se=nn.Sequential(nn.Linear(d,r),nn.ReLU(),nn.Linear(r,d),nn.Sigmoid()); self.ln=nn.Linear(d,d)
    def forward(self,x): return self.ln(x)*self.se(x)
class GatedLinear(nn.Module):
    def __init__(self,d): super().__init__(); self.fc=nn.Linear(d,2*d); self.bn=nn.BatchNorm1d(d)
    def forward(self,x): v,g=self.fc(x).chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))
class Bottleneck(nn.Module):
    def __init__(self,d): super().__init__(); m=max(8,d//4); self.op=nn.Sequential(nn.Linear(d,m),nn.ReLU(),nn.Linear(m,d),nn.ReLU())
    def forward(self,x): return self.op(x)
class LinearReLU(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS={'linear_relu':LinearReLU,'bottleneck':Bottleneck,'residual':Residual,'se_block':SEBlock,'gated_linear':GatedLinear}

class FinalDARTS(nn.Module):
    def __init__(self,arch,inp,h=512,c=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(inp,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(0.02))
        self.cell1=OPS[arch['cell1']['best_op']](h)
        self.cell2=OPS[arch['cell2']['best_op']](h)
        self.cell3=OPS[arch['cell3']['best_op']](h)
        self.cell4=OPS[arch['cell4']['best_op']](h)
        self.head=nn.Sequential(nn.LayerNorm(h),nn.Dropout(0.05),nn.Linear(h,c))
    def forward(self,x):
        x=self.proj(x); x=self.cell1(x); x=self.cell2(x); x=self.cell3(x); x=self.cell4(x); return self.head(x)

model=FinalDARTS(best_arch,input_dim,512,num_classes).to(device)
criterion=nn.CrossEntropyLoss(weight=cw_tensor, label_smoothing=0.0)
opt=optim.AdamW(model.parameters(),lr=2e-4,weight_decay=5e-5)
sch=optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=4)

EPOCHS=60; BATCH=512; PATIENCE=12
best_f1=0.0; pat=0

for epoch in range(EPOCHS):
    model.train()
    idx=np.random.permutation(len(X_retrain))
    for i in range(0,len(idx),BATCH):
        xb=torch.FloatTensor(X_retrain[idx[i:i+BATCH]]).to(device)
        yb=torch.LongTensor(y_retrain[idx[i:i+BATCH]]).to(device)
        opt.zero_grad()
        loss=criterion(model(xb),yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()

    model.eval(); preds=[]; probs=[]
    with torch.no_grad():
        for i in range(0,len(X_test),1024):
            xb=torch.FloatTensor(X_test[i:i+1024]).to(device)
            lg=model(xb)
            preds.extend(lg.argmax(1).cpu().numpy())
            probs.extend(torch.softmax(lg,1).cpu().numpy())

    preds=np.array(preds); probs=np.array(probs)
    f1=f1_score(y_test,preds,average='macro',zero_division=0)
    acc=accuracy_score(y_test,preds)
    prec=precision_score(y_test,preds,average='macro',zero_division=0)
    rec=recall_score(y_test,preds,average='macro',zero_division=0)
    mcc=matthews_corrcoef(y_test,preds)
    auc=roc_auc_score(y_test,probs,multi_class='ovr',average='macro')
    print(f"Ep{epoch+1:03d} | F1:{f1:.4f} Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} MCC:{mcc:.4f} AUC:{auc:.4f}")

    sch.step(f1)
    if f1>best_f1:
        best_f1=f1; pat=0
        torch.save(model.state_dict(),'paper_results/best_final_model.pth')
    else:
        pat+=1
        if pat>=PATIENCE: break

print("Best F1:",best_f1)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pickle, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve

# -----------------------------
# 1) Load data + best arch
# -----------------------------
with open('paper_results/block1_data.pkl','rb') as f:
    data = pickle.load(f)
with open('paper_results/best_architecture.json','r') as f:
    best_arch = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = data['num_classes']
input_dim   = data['input_dim']
X_test      = data['X_test']
y_test      = data['y_test']

# -----------------------------
# 2) Rebuild model (same as Block-3)
# -----------------------------
class Residual(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.BatchNorm1d(d),nn.ReLU())
    def forward(self,x): return x+self.op(x)

class SEBlock(nn.Module):
    def __init__(self,d):
        super().__init__()
        r=max(4,d//8)
        self.se=nn.Sequential(nn.Linear(d,r),nn.ReLU(),nn.Linear(r,d),nn.Sigmoid())
        self.ln=nn.Linear(d,d)
    def forward(self,x): return self.ln(x)*self.se(x)

class GatedLinear(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.fc=nn.Linear(d,2*d); self.bn=nn.BatchNorm1d(d)
    def forward(self,x):
        v,g=self.fc(x).chunk(2,dim=-1)
        return self.bn(v*torch.sigmoid(g))

class Bottleneck(nn.Module):
    def __init__(self,d):
        super().__init__()
        m=max(8,d//4)
        self.op=nn.Sequential(nn.Linear(d,m),nn.ReLU(),nn.Linear(m,d),nn.ReLU())
    def forward(self,x): return self.op(x)

class LinearReLU(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS={'linear_relu':LinearReLU,'bottleneck':Bottleneck,'residual':Residual,'se_block':SEBlock,'gated_linear':GatedLinear}

class FinalDARTS(nn.Module):
    def __init__(self,arch,inp,h=512,c=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(inp,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(0.02))
        self.cell1=OPS[arch['cell1']['best_op']](h)
        self.cell2=OPS[arch['cell2']['best_op']](h)
        self.cell3=OPS[arch['cell3']['best_op']](h)
        self.cell4=OPS[arch['cell4']['best_op']](h)
        self.head=nn.Sequential(nn.LayerNorm(h),nn.Dropout(0.05),nn.Linear(h,c))
    def forward(self,x):
        x=self.proj(x); x=self.cell1(x); x=self.cell2(x); x=self.cell3(x); x=self.cell4(x)
        return self.head(x)

model = FinalDARTS(best_arch,input_dim,512,num_classes).to(device)
model.load_state_dict(torch.load('paper_results/best_final_model.pth', map_location=device))
model.eval()

# -----------------------------
# 3) Predict
# -----------------------------
preds=[]; probs=[]
with torch.no_grad():
    for i in range(0,len(X_test),1024):
        xb=torch.FloatTensor(X_test[i:i+1024]).to(device)
        lg=model(xb)
        preds.extend(lg.argmax(1).cpu().numpy())
        probs.extend(torch.softmax(lg,dim=1).cpu().numpy())

preds=np.array(preds); probs=np.array(probs)

# -----------------------------
# 4) Metrics table
# -----------------------------
f1  = f1_score(y_test,preds,average='macro',zero_division=0)
acc = accuracy_score(y_test,preds)
prec= precision_score(y_test,preds,average='macro',zero_division=0)
rec = recall_score(y_test,preds,average='macro',zero_division=0)
mcc = matthews_corrcoef(y_test,preds)
auc = roc_auc_score(y_test,probs,multi_class='ovr',average='macro')

summary = pd.DataFrame([{
    "F1_macro": f1,
    "Accuracy": acc,
    "Precision_macro": prec,
    "Recall_macro": rec,
    "MCC": mcc,
    "AUC_macro_ovr": auc
}])

print("\n=== FINAL METRICS TABLE ===")
display(summary.style.format("{:.4f}"))

# per-class table
report = classification_report(y_test,preds,output_dict=True,zero_division=0)
report_df = pd.DataFrame(report).T
print("\n=== PER-CLASS TABLE ===")
display(report_df.style.format("{:.4f}"))

# -----------------------------
# 5) Confusion matrix
# -----------------------------
cm = confusion_matrix(y_test,preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

# -----------------------------
# 6) ROC curves (per-class + micro)
# -----------------------------
y_bin = label_binarize(y_test, classes=list(range(num_classes)))

plt.figure(figsize=(7,6))
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_bin[:,i], probs[:,i])
    plt.plot(fpr, tpr, lw=1, alpha=0.5, label=f"Class {i}")
fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), probs.ravel())
plt.plot(fpr_micro, tpr_micro, 'k-', lw=2.5, label='Micro-average')
plt.plot([0,1],[0,1],'k--',alpha=0.5)
plt.title("ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# -----------------------------
# 7) PR curves (per-class + micro)
# -----------------------------
plt.figure(figsize=(7,6))
for i in range(num_classes):
    p, r, _ = precision_recall_curve(y_bin[:,i], probs[:,i])
    plt.plot(r, p, lw=1, alpha=0.5, label=f"Class {i}")
p_micro, r_micro, _ = precision_recall_curve(y_bin.ravel(), probs.ravel())
plt.plot(r_micro, p_micro, 'k-', lw=2.5, label='Micro-average')
plt.title("Precision-Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# -----------------------------
# 8) Calibration curve
# -----------------------------
frac_pos, mean_pred = calibration_curve(y_bin.ravel(), probs.ravel(), n_bins=10)
plt.figure(figsize=(6,6))
plt.plot(mean_pred, frac_pos, marker='o', label='Model')
plt.plot([0,1],[0,1],'k--', label='Perfectly calibrated')
plt.title("Calibration Curve")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import torch, torch.nn as nn, numpy as np, pickle, json
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, matthews_corrcoef, roc_auc_score

# load data + arch
with open('paper_results/block1_data.pkl','rb') as f: data=pickle.load(f)
with open('paper_results/best_architecture.json','r') as f: best_arch=json.load(f)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes=data['num_classes']; input_dim=data['input_dim']
class_names=data['class_names']
X_test=data['X_test']; y_test=data['y_test']

# model def (same as your final block)
class Residual(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.BatchNorm1d(d),nn.ReLU())
    def forward(self,x): return x+self.op(x)
class SEBlock(nn.Module):
    def __init__(self,d):
        super().__init__(); r=max(4,d//8)
        self.se=nn.Sequential(nn.Linear(d,r),nn.ReLU(),nn.Linear(r,d),nn.Sigmoid()); self.ln=nn.Linear(d,d)
    def forward(self,x): return self.ln(x)*self.se(x)
class GatedLinear(nn.Module):
    def __init__(self,d): super().__init__(); self.fc=nn.Linear(d,2*d); self.bn=nn.BatchNorm1d(d)
    def forward(self,x): v,g=self.fc(x).chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))
class Bottleneck(nn.Module):
    def __init__(self,d): super().__init__(); m=max(8,d//4); self.op=nn.Sequential(nn.Linear(d,m),nn.ReLU(),nn.Linear(m,d),nn.ReLU())
    def forward(self,x): return self.op(x)
class LinearReLU(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS={'linear_relu':LinearReLU,'bottleneck':Bottleneck,'residual':Residual,'se_block':SEBlock,'gated_linear':GatedLinear}

class FinalDARTS(nn.Module):
    def __init__(self,arch,inp,h=512,c=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(inp,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(0.02))
        self.cell1=OPS[arch['cell1']['best_op']](h)
        self.cell2=OPS[arch['cell2']['best_op']](h)
        self.cell3=OPS[arch['cell3']['best_op']](h)
        self.cell4=OPS[arch['cell4']['best_op']](h)
        self.head=nn.Sequential(nn.LayerNorm(h),nn.Dropout(0.05),nn.Linear(h,c))
    def forward(self,x):
        x=self.proj(x); x=self.cell1(x); x=self.cell2(x); x=self.cell3(x); x=self.cell4(x)
        return self.head(x)

model=FinalDARTS(best_arch,input_dim,512,num_classes).to(device)
model.load_state_dict(torch.load('paper_results/best_final_model.pth', map_location=device))
model.eval()

preds=[]; probs=[]
with torch.no_grad():
    for i in range(0,len(X_test),1024):
        xb=torch.FloatTensor(X_test[i:i+1024]).to(device)
        logits=model(xb)
        preds.extend(logits.argmax(1).cpu().numpy())
        probs.extend(torch.softmax(logits,dim=1).cpu().numpy())

preds=np.array(preds); probs=np.array(probs)

f1_macro=f1_score(y_test,preds,average='macro',zero_division=0)
f1_weighted=f1_score(y_test,preds,average='weighted',zero_division=0)
f1_micro=f1_score(y_test,preds,average='micro',zero_division=0)
precision_macro=precision_score(y_test,preds,average='macro',zero_division=0)
recall_macro=recall_score(y_test,preds,average='macro',zero_division=0)
accuracy=accuracy_score(y_test,preds)
mcc=matthews_corrcoef(y_test,preds)

f1_pc=f1_score(y_test,preds,average=None,zero_division=0)
p_pc=precision_score(y_test,preds,average=None,zero_division=0)
r_pc=recall_score(y_test,preds,average=None,zero_division=0)

fp32_kb = sum(p.numel() for p in model.parameters())*4/1024

out={
    "f1_macro":float(f1_macro),
    "f1_weighted":float(f1_weighted),
    "f1_micro":float(f1_micro),
    "precision_macro":float(precision_macro),
    "recall_macro":float(recall_macro),
    "accuracy":float(accuracy),
    "mcc":float(mcc),
    "hidden_dim":512,
    "fp32_kb":float(fp32_kb),
    "f1_per_class":{c:float(f1_pc[i]) for i,c in enumerate(class_names)},
    "precision_per_class":{c:float(p_pc[i]) for i,c in enumerate(class_names)},
    "recall_per_class":{c:float(r_pc[i]) for i,c in enumerate(class_names)}
}

with open('paper_results/final_results.json','w') as f:
    json.dump(out,f,indent=2)

print("✅ saved: paper_results/final_results.json")
print(f"F1_macro={f1_macro:.4f}, Acc={accuracy:.4f}, MCC={mcc:.4f}, Size={fp32_kb:.1f}KB")

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
import numpy as np, pickle, json, os, time, copy, gc
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                              recall_score, matthews_corrcoef)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

print("="*70)
print("BLOCK 4: Baseline Comparison — STRICT (final)")
print("="*70)

# ---------------- STRICT LOADS ----------------
with open('paper_results/block1_data.pkl','rb') as f: data=pickle.load(f)
with open('paper_results/final_results.json','r') as f: darts_r=json.load(f)
with open('paper_results/best_architecture.json','r') as f: best_arch=json.load(f)

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = data['num_classes']; input_dim   = data['input_dim']
class_names = data['class_names']; train_counts= data['train_counts']

# fixed: recompute cbrt_w instead of reading missing key
cbrt_w = np.cbrt(train_counts.max() / train_counts)
cbrt_w = cbrt_w / cbrt_w.mean()

X_retrain   = data['X_retrain']; y_retrain = data['y_retrain']
X_test      = data['X_test'];    y_test    = data['y_test']
cw_tensor   = torch.FloatTensor(cbrt_w).to(device)
BATCH       = 4096

print(f"Train:{len(X_retrain):,} | Test:{len(X_test):,} | Device:{device}")

# ── ALL OPS DEFINED HERE ────────────────────
class LinearReLU(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.ReLU())
    def forward(self,x): return self.op(x)
class LinearBNReLU(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.ReLU())
    def forward(self,x): return self.op(x)
class LinearDropReLU(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.Dropout(0.3),nn.ReLU())
    def forward(self,x): return self.op(x)
class Bottleneck(nn.Module):
    def __init__(self,dim):
        super().__init__(); mid=max(16,dim//4)
        self.op=nn.Sequential(nn.Linear(dim,mid),nn.ReLU(),nn.Linear(mid,dim),nn.ReLU())
    def forward(self,x): return self.op(x)
class Skip(nn.Module):
    def __init__(self,dim): super().__init__()
    def forward(self,x): return x
class LinearBNDrop(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.ReLU(),nn.Dropout(0.2))
    def forward(self,x): return self.op(x)
class Residual(nn.Module):
    def __init__(self,dim): super().__init__(); self.op=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.ReLU())
    def forward(self,x): return x+self.op(x)
class SEBlock(nn.Module):
    def __init__(self,dim):
        super().__init__(); r=max(8,dim//8)
        self.se=nn.Sequential(nn.Linear(dim,r),nn.ReLU(),nn.Linear(r,dim),nn.Sigmoid())
        self.ln=nn.Linear(dim,dim)
    def forward(self,x): return self.ln(x)*self.se(x)
class GatedLinear(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.fc=nn.Linear(dim,dim*2); self.bn=nn.BatchNorm1d(dim)
    def forward(self,x):
        h=self.fc(x); v,g=h.chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))

OPS={
    'linear_relu':LinearReLU,'linear_bn_relu':LinearBNReLU,
    'linear_drop':LinearDropReLU,'bottleneck':Bottleneck,'skip':Skip,
    'linear_bn_drop':LinearBNDrop,'residual':Residual,
    'se_block':SEBlock,'gated_linear':GatedLinear,
}

# ── DARTS model definition ────────────────────────────────────
class FinalDARTS(nn.Module):
    def __init__(self,arch,input_dim,hidden=256,num_classes=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(input_dim,hidden),nn.BatchNorm1d(hidden),nn.ReLU(),nn.Dropout(0.1))
        self.cell1=OPS[arch['cell1']['best_op']](hidden)
        self.cell2=OPS[arch['cell2']['best_op']](hidden)
        self.cell3=OPS[arch['cell3']['best_op']](hidden)
        self.cell4=OPS[arch['cell4']['best_op']](hidden)
        r=max(8,hidden//8)
        self.attn=nn.Sequential(nn.Linear(hidden,r),nn.ReLU(),nn.Linear(r,hidden),nn.Sigmoid())
        self.head=nn.Sequential(nn.LayerNorm(hidden),nn.Dropout(0.2),nn.Linear(hidden,num_classes))
    def forward(self,x):
        x=self.proj(x)
        x=self.cell1(x); x=self.cell2(x)
        x=self.cell3(x); x=self.cell4(x)
        x=x*self.attn(x); return self.head(x)

def get_metrics(y_true,y_pred,name=''):
    return {
        'name':name,
        'f1_macro':     f1_score(y_true,y_pred,average='macro',    zero_division=0),
        'f1_weighted':  f1_score(y_true,y_pred,average='weighted', zero_division=0),
        'f1_micro':     f1_score(y_true,y_pred,average='micro',    zero_division=0),
        'precision':    precision_score(y_true,y_pred,average='macro',zero_division=0),
        'recall':       recall_score(y_true,y_pred,average='macro',   zero_division=0),
        'accuracy':     accuracy_score(y_true,y_pred),
        'mcc':          matthews_corrcoef(y_true,y_pred),
        'f1_per_class': f1_score(y_true,y_pred,average=None,zero_division=0),
        'prec_per_class':precision_score(y_true,y_pred,average=None,zero_division=0),
        'rec_per_class': recall_score(y_true,y_pred,average=None,   zero_division=0),
    }

all_results={}

# ── 1. DARTS ──────────────────────────────────────────────────
print("\n── 1. DARTS-IDS ──")
all_results['DARTS-IDS (Ours)']={
    'f1_macro':darts_r['f1_macro'],'f1_weighted':darts_r['f1_weighted'],
    'f1_micro':darts_r['f1_micro'],'precision':darts_r['precision_macro'],
    'recall':darts_r['recall_macro'],'accuracy':darts_r['accuracy'],
    'mcc':darts_r['mcc'],'size_kb':darts_r['fp32_kb'],'train_n':len(X_retrain),
    'f1_per_class': np.array([darts_r['f1_per_class'][c]         for c in class_names]),
    'prec_per_class':np.array([darts_r['precision_per_class'][c] for c in class_names]),
    'rec_per_class': np.array([darts_r['recall_per_class'][c]    for c in class_names]),
}
print(f"   F1:{darts_r['f1_macro']:.4f} Acc:{darts_r['accuracy']:.4f} MCC:{darts_r['mcc']:.4f}")

# ── 2 & 3. MLP ────────────────────────────────────────────────
class BaselineMLP(nn.Module):
    def __init__(self,idim,hidden=256,nc=10):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(idim,hidden),nn.BatchNorm1d(hidden),nn.ReLU(),
            nn.Linear(hidden,hidden),nn.BatchNorm1d(hidden),nn.ReLU(),
            nn.Linear(hidden,hidden),nn.BatchNorm1d(hidden),nn.ReLU(),
            nn.Dropout(0.2),nn.Linear(hidden,nc))
    def forward(self,x): return self.net(x)

def train_mlp(use_cw,label,epochs=100):
    print(f"\n── {label} ──")
    model=BaselineMLP(input_dim,256,num_classes).to(device)
    crit=(nn.CrossEntropyLoss(weight=cw_tensor,label_smoothing=0.01)
          if use_cw else nn.CrossEntropyLoss(label_smoothing=0.01))
    opt=optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=1e-6)
    best_f1=0; best_state=None; pat=0; t0=time.time()
    for ep in range(epochs):
        model.train()
        perm=np.random.permutation(len(X_retrain))
        for step in range(len(X_retrain)//BATCH):
            i=step*BATCH
            xb=torch.FloatTensor(X_retrain[perm[i:i+BATCH]]).to(device)
            yb=torch.LongTensor(y_retrain[perm[i:i+BATCH]]).to(device)
            opt.zero_grad(); crit(model(xb),yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); del xb,yb
        sch.step(); torch.cuda.empty_cache()
        if (ep+1)%10==0 or ep==epochs-1:
            model.eval(); preds=[]
            with torch.no_grad():
                for i in range(0,len(X_test),BATCH*4):
                    xb=torch.FloatTensor(X_test[i:i+BATCH*4]).to(device)
                    preds.extend(model(xb).argmax(1).cpu().numpy()); del xb
            f1=f1_score(y_test,preds,average='macro',zero_division=0)
            acc=accuracy_score(y_test,preds)
            print(f"   Ep{ep+1:03d} | F1:{f1:.4f} Acc:{acc:.4f}")
            if f1>best_f1: best_f1=f1; best_state=copy.deepcopy(model.state_dict()); pat=0
            else: pat+=1
            if pat>=4: break
    model.load_state_dict(best_state); model.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,len(X_test),BATCH*4):
            xb=torch.FloatTensor(X_test[i:i+BATCH*4]).to(device)
            preds.extend(model(xb).argmax(1).cpu().numpy()); del xb
    m=get_metrics(y_test,np.array(preds),label)
    m['size_kb']=sum(p.numel() for p in model.parameters())*4/1024
    m['train_n']=len(X_retrain)
    print(f"   DONE | F1:{m['f1_macro']:.4f} Acc:{m['accuracy']:.4f} | {time.time()-t0:.0f}s")
    del model; torch.cuda.empty_cache(); gc.collect()
    return m

all_results['MLP + Class Weights']  = train_mlp(True,  '2. MLP + Class Weights')
all_results['MLP No Class Weights'] = train_mlp(False, '3. MLP No Class Weights')

# ── 4. XGBoost ────────────────────────────────────────────────
print("\n── 4. XGBoost ──")
try:
    import xgboost as xgb, pickle as pkl
    xgb_clf=xgb.XGBClassifier(
        n_estimators=300,max_depth=7,learning_rate=0.1,
        subsample=0.8,colsample_bytree=0.8,
        objective='multi:softprob',eval_metric='mlogloss',
        use_label_encoder=False,verbosity=0,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        tree_method='hist',n_jobs=-1,random_state=42,early_stopping_rounds=20)
    t0=time.time()
    xgb_clf.fit(X_retrain,y_retrain,
                eval_set=[(X_test[:50000],y_test[:50000])],verbose=50)
    xgb_pred=xgb_clf.predict(X_test)
    xgb_size=len(pkl.dumps(xgb_clf))/1024
    m=get_metrics(y_test,xgb_pred,'XGBoost')
    m['size_kb']=xgb_size; m['train_n']=len(X_retrain)
    all_results['XGBoost']=m
    print(f"   F1:{m['f1_macro']:.4f} Acc:{m['accuracy']:.4f} Size:{xgb_size:.0f}KB | {time.time()-t0:.0f}s")
    xgb_clf.save_model('paper_results/xgboost_model.json')
    del xgb_clf; gc.collect()
except Exception as e: print(f"   Error: {e}")

# ── 5. LightGBM ───────────────────────────────────────────────
print("\n── 5. LightGBM ──")
try:
    import lightgbm as lgb, pickle as pkl
    lgb_clf=lgb.LGBMClassifier(
        n_estimators=300,max_depth=7,learning_rate=0.1,
        num_leaves=63,subsample=0.8,colsample_bytree=0.8,
        objective='multiclass',n_jobs=-1,random_state=42,verbose=-1)
    t0=time.time()
    lgb_clf.fit(X_retrain,y_retrain,
                eval_set=[(X_test[:50000],y_test[:50000])],
                callbacks=[lgb.early_stopping(20,verbose=False),lgb.log_evaluation(50)])
    lgb_pred=lgb_clf.predict(X_test)
    lgb_size=len(pkl.dumps(lgb_clf))/1024
    m=get_metrics(y_test,lgb_pred,'LightGBM')
    m['size_kb']=lgb_size; m['train_n']=len(X_retrain)
    all_results['LightGBM']=m
    print(f"   F1:{m['f1_macro']:.4f} Acc:{m['accuracy']:.4f} Size:{lgb_size:.0f}KB | {time.time()-t0:.0f}s")
    del lgb_clf; gc.collect()
except Exception as e: print(f"   Error: {e}")

# ── 6. Random Forest ──────────────────────────────────────────
print("\n── 6. Random Forest (2M subsample) ──")
import pickle as pkl
rng=np.random.RandomState(42); rf_idx=[]
for c in range(num_classes):
    c_idx=np.where(y_retrain==c)[0]
    n_c=max(500,int(2_000_000*len(c_idx)/len(y_retrain)))
    rf_idx.extend(rng.choice(c_idx,min(n_c,len(c_idx)),replace=False))
rf_idx=np.array(rf_idx); X_rf=X_retrain[rf_idx]; y_rf=y_retrain[rf_idx]
print(f"   Subsample: {len(X_rf):,}")
rf_clf=RandomForestClassifier(n_estimators=200,max_depth=20,min_samples_leaf=5,
    max_features='sqrt',class_weight='balanced_subsample',n_jobs=-1,random_state=42)
t0=time.time(); rf_clf.fit(X_rf,y_rf); rf_pred=rf_clf.predict(X_test)
rf_size=len(pkl.dumps(rf_clf))/1024
m=get_metrics(y_test,rf_pred,'Random Forest')
m['size_kb']=rf_size; m['train_n']=len(X_rf)
all_results['Random Forest']=m
print(f"   F1:{m['f1_macro']:.4f} Acc:{m['accuracy']:.4f} | {time.time()-t0:.0f}s")
del rf_clf; gc.collect()

# ── 7. Logistic Regression ────────────────────────────────────
print("\n── 7. Logistic Regression (500K subsample) ──")
lr_idx=rng.choice(len(X_rf),min(500_000,len(X_rf)),replace=False)
X_lr=X_rf[lr_idx]; y_lr=y_rf[lr_idx]
lr_clf=LogisticRegression(C=1.0,max_iter=500,solver='saga',
    class_weight='balanced',n_jobs=-1,random_state=42,tol=1e-3)
t0=time.time(); lr_clf.fit(X_lr,y_lr); lr_pred=lr_clf.predict(X_test)
lr_size=len(pkl.dumps(lr_clf))/1024
m=get_metrics(y_test,lr_pred,'Logistic Reg.')
m['size_kb']=lr_size; m['train_n']=len(X_lr)
all_results['Logistic Reg.']=m
print(f"   F1:{m['f1_macro']:.4f} Acc:{m['accuracy']:.4f} | {time.time()-t0:.0f}s")

# ── Noise Robustness ──────────────────────────────────────────
print("\n── Noise Robustness Test ──")
darts_model=FinalDARTS(best_arch,input_dim,darts_r['hidden_dim'],num_classes).to(device)
darts_model.load_state_dict(torch.load('paper_results/best_final_model.pth',map_location=device))
darts_model.eval()

has_xgb=os.path.exists('paper_results/xgboost_model.json')
if has_xgb:
    import xgboost as xgb
    xgb_r=xgb.XGBClassifier(); xgb_r.load_model('paper_results/xgboost_model.json')

robustness={}
for sigma in [0.0,0.05,0.1,0.2,0.5]:
    np.random.seed(42)
    X_noisy=(X_test+np.random.normal(0,sigma,X_test.shape)).astype(np.float32)
    d_preds=[]
    with torch.no_grad():
        for i in range(0,len(X_noisy),BATCH*4):
            xb=torch.FloatTensor(X_noisy[i:i+BATCH*4]).to(device)
            d_preds.extend(darts_model(xb).argmax(1).cpu().numpy()); del xb
    f1_d=f1_score(y_test,np.array(d_preds),average='macro',zero_division=0)
    f1_x=f1_score(y_test,xgb_r.predict(X_noisy),average='macro',zero_division=0) if has_xgb else 0.0
    robustness[sigma]={'DARTS':f1_d,'XGBoost':f1_x}
    print(f"   σ={sigma:.2f} | DARTS:{f1_d:.4f} | XGB:{f1_x:.4f}")
    del X_noisy; gc.collect()

del darts_model; torch.cuda.empty_cache(); gc.collect()

# ── Final Table ───────────────────────────────────────────────
order=['DARTS-IDS (Ours)','MLP + Class Weights','MLP No Class Weights',
       'XGBoost','LightGBM','Random Forest','Logistic Reg.']
present=[n for n in order if n in all_results]
ESP32=8192

print(f"\n{'='*115}")
print("FINAL COMPARISON TABLE")
print(f"{'='*115}")
print(f"  {'Method':<25} {'F1-mac':>7} {'F1-wt':>7} {'Acc':>7} {'Prec':>7} "
      f"{'Rec':>7} {'MCC':>7} {'SizeKB':>10} {'ESP32?':>7}")
print(f"  {'─'*110}")
for name in present:
    r=all_results[name]
    esp='✅' if r['size_kb']<ESP32 else '❌'
    print(f"  {name:<25} {r['f1_macro']:>7.4f} {r['f1_weighted']:>7.4f} "
          f"{r['accuracy']:>7.4f} {r['precision']:>7.4f} {r['recall']:>7.4f} "
          f"{r['mcc']:>7.4f} {r['size_kb']:>10.1f} {esp:>7}")

# ── Save ──────────────────────────────────────────────────────
save={}
for name,r in all_results.items():
    save[name]={k:(float(v) if isinstance(v,(float,np.floating)) else
                   int(v)   if isinstance(v,(int,np.integer))    else
                   {c:float(v[i]) for i,c in enumerate(class_names)} if isinstance(v,np.ndarray) else v)
                for k,v in r.items()}
with open('paper_results/all_baselines.json','w') as f: json.dump(save,f,indent=2)
with open('paper_results/robustness.json','w') as f:
    json.dump({str(k):v for k,v in robustness.items()},f,indent=2)
print("\n✅ Block 4 complete")

In [ ]:
import torch, torch.nn as nn, json, pickle

# load metadata
with open('paper_results/block1_data.pkl','rb') as f: data=pickle.load(f)
with open('paper_results/best_architecture.json','r') as f: best_arch=json.load(f)
with open('paper_results/final_results.json','r') as f: darts_r=json.load(f)

num_classes=data['num_classes']
input_dim=data['input_dim']
HIDDEN=darts_r.get('hidden_dim',512)

# --- exact ops used by checkpoint ---
class Residual(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.BatchNorm1d(d),nn.ReLU())
    def forward(self,x): return x+self.op(x)

class SEBlock(nn.Module):
    def __init__(self,d):
        super().__init__(); r=max(4,d//8)
        self.se=nn.Sequential(nn.Linear(d,r),nn.ReLU(),nn.Linear(r,d),nn.Sigmoid()); self.ln=nn.Linear(d,d)
    def forward(self,x): return self.ln(x)*self.se(x)

class GatedLinear(nn.Module):
    def __init__(self,d): super().__init__(); self.fc=nn.Linear(d,2*d); self.bn=nn.BatchNorm1d(d)
    def forward(self,x): v,g=self.fc(x).chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))

class Bottleneck(nn.Module):
    def __init__(self,d): super().__init__(); m=max(8,d//4); self.op=nn.Sequential(nn.Linear(d,m),nn.ReLU(),nn.Linear(m,d),nn.ReLU())
    def forward(self,x): return self.op(x)

class LinearReLU(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS={'linear_relu':LinearReLU,'bottleneck':Bottleneck,'residual':Residual,'se_block':SEBlock,'gated_linear':GatedLinear}

class FinalDARTS(nn.Module):
    def __init__(self,arch,inp,h=512,c=10):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(inp,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(0.02))
        self.cell1=OPS[arch['cell1']['best_op']](h)
        self.cell2=OPS[arch['cell2']['best_op']](h)
        self.cell3=OPS[arch['cell3']['best_op']](h)
        self.cell4=OPS[arch['cell4']['best_op']](h)
        self.head=nn.Sequential(nn.LayerNorm(h),nn.Dropout(0.05),nn.Linear(h,c))
    def forward(self,x):
        x=self.proj(x); x=self.cell1(x); x=self.cell2(x); x=self.cell3(x); x=self.cell4(x)
        return self.head(x)

# load checkpoint
model_fp32=FinalDARTS(best_arch,input_dim,HIDDEN,num_classes).cpu()
state=torch.load('paper_results/best_final_model.pth',map_location='cpu')
model_fp32.load_state_dict(state, strict=True)
model_fp32.eval()

print("✅ Checkpoint loaded successfully with exact architecture.")
print(f"Params: {sum(p.numel() for p in model_fp32.parameters()):,}")

In [ ]:
import torch, torch.nn as nn, json, pickle

with open('paper_results/block1_data.pkl','rb') as f: data=pickle.load(f)
with open('paper_results/best_architecture.json','r') as f: best_arch=json.load(f)

num_classes = data['num_classes']
input_dim   = data['input_dim']

state = torch.load('paper_results/best_final_model.pth', map_location='cpu')

# infer hidden dim from checkpoint directly
# proj.0.weight shape = [hidden, input_dim]
hidden = state['proj.0.weight'].shape[0]
# infer output classes from final linear
num_cls_ckpt = state['head.2.weight'].shape[0]

print(f"Inferred hidden={hidden}, classes={num_cls_ckpt}")

class Residual(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.BatchNorm1d(d),nn.ReLU())
    def forward(self,x): return x+self.op(x)
class SEBlock(nn.Module):
    def __init__(self,d):
        super().__init__(); r=max(4,d//8)
        self.se=nn.Sequential(nn.Linear(d,r),nn.ReLU(),nn.Linear(r,d),nn.Sigmoid()); self.ln=nn.Linear(d,d)
    def forward(self,x): return self.ln(x)*self.se(x)
class GatedLinear(nn.Module):
    def __init__(self,d): super().__init__(); self.fc=nn.Linear(d,2*d); self.bn=nn.BatchNorm1d(d)
    def forward(self,x): v,g=self.fc(x).chunk(2,dim=-1); return self.bn(v*torch.sigmoid(g))
class Bottleneck(nn.Module):
    def __init__(self,d): super().__init__(); m=max(8,d//4); self.op=nn.Sequential(nn.Linear(d,m),nn.ReLU(),nn.Linear(m,d),nn.ReLU())
    def forward(self,x): return self.op(x)
class LinearReLU(nn.Module):
    def __init__(self,d): super().__init__(); self.op=nn.Sequential(nn.Linear(d,d),nn.ReLU())
    def forward(self,x): return self.op(x)

OPS={'linear_relu':LinearReLU,'bottleneck':Bottleneck,'residual':Residual,'se_block':SEBlock,'gated_linear':GatedLinear}

class FinalDARTS(nn.Module):
    def __init__(self,arch,inp,h,c):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(inp,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(0.02))
        self.cell1=OPS[arch['cell1']['best_op']](h)
        self.cell2=OPS[arch['cell2']['best_op']](h)
        self.cell3=OPS[arch['cell3']['best_op']](h)
        self.cell4=OPS[arch['cell4']['best_op']](h)
        self.head=nn.Sequential(nn.LayerNorm(h),nn.Dropout(0.05),nn.Linear(h,c))
    def forward(self,x):
        x=self.proj(x); x=self.cell1(x); x=self.cell2(x); x=self.cell3(x); x=self.cell4(x)
        return self.head(x)

model_fp32 = FinalDARTS(best_arch, input_dim, hidden, num_cls_ckpt).cpu()
missing, unexpected = model_fp32.load_state_dict(state, strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)

# hard check
assert len(missing)==0 and len(unexpected)==0, "Still mismatched. Share printed keys."
model_fp32.eval()
print("✅ Model loaded successfully.")

In [ ]:
# Code: Class-aware quantization search (no retrain)
import copy, torch, torch.nn as nn, numpy as np, json, os

os.makedirs("paper_results", exist_ok=True)

# 1) baseline metrics
fp32_metrics = eval_cpu(model_fp32, X_test, y_test)
fp32_kb = model_kb(model_fp32, 'paper_results/model_fp32.pth')
t_fp32,_ = infer_ms(model_fp32, X_test)

# hard classes = low FP32 F1 classes (class-aware target)
hard_idx = [i for i,v in enumerate(fp32_metrics['f1_per_class']) if v < 0.85]
if len(hard_idx)==0:
    hard_idx = list(np.argsort(fp32_metrics['f1_per_class'])[:max(2, num_classes//5)])
hard_names = [class_names[i] for i in hard_idx]
print("Hard classes:", hard_names)

# 2) quantized base
q_base = torch.quantization.quantize_dynamic(copy.deepcopy(model_fp32), {nn.Linear}, dtype=torch.qint8).cpu().eval()
q_metrics = eval_cpu(q_base, X_test, y_test)
q_kb = model_kb(q_base, 'paper_results/model_int8_all.pth')
t_q,_ = infer_ms(q_base, X_test)

print(f"FP32      F1={fp32_metrics['f1_macro']:.4f} Acc={fp32_metrics['accuracy']:.4f} Size={fp32_kb:.1f}KB")
print(f"INT8_all  F1={q_metrics['f1_macro']:.4f} Acc={q_metrics['accuracy']:.4f} Size={q_kb:.1f}KB")

# helper: hard-class mean f1
def hard_f1_mean(m):
    arr = np.array(m['f1_per_class'])
    return float(arr[hard_idx].mean())

# 3) candidate modules to restore
candidates = ['head', 'proj', 'cell1', 'cell2', 'cell3', 'cell4']

def build_variant(restored):
    m = torch.quantization.quantize_dynamic(copy.deepcopy(model_fp32), {nn.Linear}, dtype=torch.qint8).cpu().eval()
    for name in restored:
        setattr(m, name, copy.deepcopy(getattr(model_fp32, name)))
    return m

# 4) greedy class-aware restore search
chosen = []
best_model = q_base
best_metrics = q_metrics
best_kb = q_kb
best_score = (hard_f1_mean(q_metrics), q_metrics['f1_macro'])

target_macro = max(0.86, fp32_metrics['f1_macro'] - 0.03)  # adjustable
target_hard  = max(0.75, hard_f1_mean(fp32_metrics) - 0.05)

print(f"Targets: macro>={target_macro:.3f}, hard_mean>={target_hard:.3f}")

while True:
    pool = [c for c in candidates if c not in chosen]
    if not pool:
        break

    trial_rows = []
    for c in pool:
        trial_restore = chosen + [c]
        m = build_variant(trial_restore)
        met = eval_cpu(m, X_test, y_test)
        kb  = model_kb(m, f'paper_results/model_trial_{"_".join(trial_restore)}.pth')

        gain_h = hard_f1_mean(met) - best_score[0]
        gain_m = met['f1_macro'] - best_score[1]
        kb_inc = kb - best_kb + 1e-9
        score  = (2.0*gain_h + 1.0*gain_m) / kb_inc   # class-aware gain per KB

        trial_rows.append((c, score, met, kb, m))

    trial_rows.sort(key=lambda x: x[1], reverse=True)
    c, sc, met, kb, m = trial_rows[0]

    # accept only if improves class-aware objective
    if (hard_f1_mean(met) > best_score[0] + 1e-4) or (met['f1_macro'] > best_score[1] + 1e-4):
        chosen.append(c)
        best_model, best_metrics, best_kb = m, met, kb
        best_score = (hard_f1_mean(met), met['f1_macro'])
        print(f" + restore {c:<6} -> F1={met['f1_macro']:.4f}, hard={hard_f1_mean(met):.4f}, size={kb:.1f}KB")
        if met['f1_macro'] >= target_macro and hard_f1_mean(met) >= target_hard:
            print("Reached target.")
            break
    else:
        print("No further positive restore. stopping.")
        break

# final CAQ
caq_model = best_model
caq_metrics = best_metrics
caq_kb = best_kb
t_caq,_ = infer_ms(caq_model, X_test)

torch.save(caq_model.state_dict(), 'paper_results/model_caq_best.pth')

print("\n=== FINAL CAQ ===")
print("Restored modules:", chosen)
print(f"CAQ_best F1={caq_metrics['f1_macro']:.4f} Acc={caq_metrics['accuracy']:.4f} hard={hard_f1_mean(caq_metrics):.4f}")
print(f"Size={caq_kb:.1f}KB  Latency={t_caq:.3f}ms")

# save report
report = {
    "hard_classes": hard_names,
    "restored_modules": chosen,
    "fp32": {"f1_macro": float(fp32_metrics['f1_macro']), "accuracy": float(fp32_metrics['accuracy']),
             "hard_f1_mean": float(hard_f1_mean(fp32_metrics)), "size_kb": float(fp32_kb), "lat_ms": float(t_fp32)},
    "int8_all": {"f1_macro": float(q_metrics['f1_macro']), "accuracy": float(q_metrics['accuracy']),
                 "hard_f1_mean": float(hard_f1_mean(q_metrics)), "size_kb": float(q_kb), "lat_ms": float(t_q)},
    "caq_best": {"f1_macro": float(caq_metrics['f1_macro']), "accuracy": float(caq_metrics['accuracy']),
                 "hard_f1_mean": float(hard_f1_mean(caq_metrics)), "size_kb": float(caq_kb), "lat_ms": float(t_caq)}
}
with open('paper_results/caq_search_report.json','w') as f:
    json.dump(report, f, indent=2)

print("✅ saved: paper_results/caq_search_report.json")

In [ ]:
# Code: install ONNX + ONNX Runtime in Kaggle
!pip -q install onnx onnxruntime onnxruntime-tools

In [ ]:
# Code: targeted nodes_to_exclude search (best F1)
import os, json, itertools, onnx, numpy as np, onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, quantize_static, QuantType, QuantFormat, CalibrationMethod
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, matthews_corrcoef

onnx_fp32 = "paper_results/darts_fp32.onnx"
X_test = data['X_test'].astype(np.float32)
y_test = data['y_test']
X_retrain = data['X_retrain'].astype(np.float32)
y_retrain = data['y_retrain']
num_classes = data['num_classes']

def make_balanced_calib(X, y, per_class=2048, seed=42):
    rng=np.random.RandomState(seed); idx=[]
    for c in range(num_classes):
        c_idx=np.where(y==c)[0]
        idx.extend(rng.choice(c_idx, size=min(per_class,len(c_idx)), replace=False).tolist())
    rng.shuffle(idx)
    return X[np.array(idx)].astype(np.float32)

class Reader(CalibrationDataReader):
    def __init__(self, X, bs=256, inp="input"):
        self.X=X; self.bs=bs; self.inp=inp; self.i=0
    def get_next(self):
        if self.i>=len(self.X): return None
        j=min(self.i+self.bs, len(self.X))
        b=self.X[self.i:j]; self.i=j
        return {self.inp:b}

def eval_onnx(path):
    sess=ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    inp=sess.get_inputs()[0].name
    preds=[]
    for i in range(0,len(X_test),4096):
        lg=sess.run(None,{inp:X_test[i:i+4096]})[0]
        preds.append(np.argmax(lg,axis=1))
    preds=np.concatenate(preds)
    return {
        "f1_macro":f1_score(y_test,preds,average='macro',zero_division=0),
        "accuracy":accuracy_score(y_test,preds),
        "precision_macro":precision_score(y_test,preds,average='macro',zero_division=0),
        "recall_macro":recall_score(y_test,preds,average='macro',zero_division=0),
        "mcc":matthews_corrcoef(y_test,preds),
    }

# list nodes
m = onnx.load(onnx_fp32)
nodes = [n.name for n in m.graph.node if n.name]
print("Nodes:", nodes)

# candidate sensitive nodes (tail + norm + mul gates)
cand = [n for n in nodes if any(k in n.lower() for k in ["layer_norm","mul","add","gemm","linear_8","linear_9","linear_10"])]
cand = sorted(list(set(cand)))
print("Candidate excludes:", cand)

X_cal = make_balanced_calib(X_retrain, y_retrain, per_class=2048)
reader_data = X_cal  # recreate reader each run

base_best = {"f1_macro": -1}
rows=[]

# try combos of size 1..3 (fast enough)
for r in [1,2,3]:
    for combo in itertools.combinations(cand, r):
        out = f"paper_results/caq_excl_{r}_{abs(hash(combo))%10**8}.onnx"
        reader = Reader(reader_data, bs=256, inp="input")
        try:
            quantize_static(
                model_input=onnx_fp32,
                model_output=out,
                calibration_data_reader=reader,
                quant_format=QuantFormat.QDQ,
                activation_type=QuantType.QUInt8,
                weight_type=QuantType.QInt8,
                calibrate_method=CalibrationMethod.Percentile,
                per_channel=True,
                nodes_to_exclude=list(combo)
            )
            met = eval_onnx(out)
            sz = os.path.getsize(out)/1024
            row = {"exclude":list(combo), "size_kb":sz, **met, "file":out}
            rows.append(row)
            if met["f1_macro"] > base_best["f1_macro"]:
                base_best = row
                print("NEW BEST:", base_best["f1_macro"], "exclude=", base_best["exclude"])
        except Exception as e:
            pass

rows = sorted(rows, key=lambda x:(x["f1_macro"], x["mcc"]), reverse=True)
print("\nTOP 10")
for rr in rows[:10]:
    print(rr["f1_macro"], rr["accuracy"], rr["size_kb"], rr["exclude"])

with open("paper_results/caq_exclude_search.json","w") as f:
    json.dump({"best":base_best,"top10":rows[:10]}, f, indent=2)

print("\n✅ saved: paper_results/caq_exclude_search.json")
print("BEST:", base_best)

In [ ]:
# Fast finalize using discovered best exclude
best_exclude = ['node_Gemm_0']

onnx_best = "paper_results/darts_caq_best.onnx"

reader = Reader(make_balanced_calib(X_retrain, y_retrain, per_class=2048), bs=256, inp="input")
quantize_static(
    model_input=onnx_fp32,
    model_output=onnx_best,
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    calibrate_method=CalibrationMethod.Percentile,
    per_channel=True,
    nodes_to_exclude=best_exclude
)

best_metrics = eval_onnx(onnx_best)
best_size_kb = os.path.getsize(onnx_best)/1024

summary = {
    "best_model": onnx_best,
    "exclude_nodes": best_exclude,
    "metrics": best_metrics,
    "size_kb": best_size_kb,
    "reference": {
        "fp32_f1": 0.9105,
        "naive_int8_f1": 0.4766,
        "caq_percentile_2048_no_exclude_f1": 0.7990196952345506
    }
}
with open("paper_results/caq_final_best.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ FINAL CAQ")
print(summary)

In [ ]:
import onnx, onnxruntime as ort
print("onnx:", onnx.__version__)
print("ort :", ort.__version__)

In [ ]:
!pip -q install onnx onnxruntime onnxscript

In [ ]:
# Code: Generate all paper tables from your finalized results
import pandas as pd
import numpy as np

# -----------------------------
# 1) Enter known results (from your logs)
# -----------------------------
fp32 = {
    "f1_macro": 0.9105,
    "accuracy": 0.9732,
    "precision_macro": np.nan,
    "recall_macro": np.nan,
    "mcc": np.nan,
    "size_kb": 5014.8
}

int8_naive = {
    "f1_macro": 0.4766,
    "accuracy": 0.4657,
    "precision_macro": np.nan,
    "recall_macro": np.nan,
    "mcc": np.nan,
    "size_kb": 1298.7
}

caq_static_no_excl = {
    "f1_macro": 0.7990196952345506,
    "accuracy": 0.8628166228859833,
    "precision_macro": 0.7985372603794583,
    "recall_macro": 0.8190794148241138,
    "mcc": 0.8270830364222383,
    "size_kb": 1334.3349609375
}

caq_final = {
    "f1_macro": 0.9002216938421311,
    "accuracy": 0.9690496148283699,
    "precision_macro": 0.9036004806467522,
    "recall_macro": 0.8995390269636658,
    "mcc": 0.9602398161300575,
    "size_kb": 1388.5419921875
}

# calibration ablation rows
calib_rows = [
    {"method":"Percentile","per_class":512,"size_kb":1334.3359375,"f1_macro":0.7845093168187496,"accuracy":0.8442743720669402,"precision_macro":0.7798771385072415,"recall_macro":0.8124520676755876,"mcc":0.8056875944671827},
    {"method":"Entropy","per_class":512,"size_kb":1334.33203125,"f1_macro":0.29720090677607514,"accuracy":0.3647581240223134,"precision_macro":0.33923881594416183,"recall_macro":0.45216001896906066,"mcc":0.2627923162110109},
    {"method":"Percentile","per_class":1024,"size_kb":1334.3369140625,"f1_macro":0.7889918496703409,"accuracy":0.8482677606918332,"precision_macro":0.7863121952791936,"recall_macro":0.814127952318328,"mcc":0.8116381527608179},
    {"method":"Entropy","per_class":1024,"size_kb":1334.33203125,"f1_macro":0.30373116833290287,"accuracy":0.36906142085534666,"precision_macro":0.34635303517741856,"recall_macro":0.4621875949958124,"mcc":0.26682309402660265},
    {"method":"Percentile","per_class":2048,"size_kb":1334.3349609375,"f1_macro":0.7990196952345506,"accuracy":0.8628166228859833,"precision_macro":0.7985372603794583,"recall_macro":0.8190794148241138,"mcc":0.8270830364222383},
    {"method":"Entropy","per_class":2048,"size_kb":1334.33203125,"f1_macro":0.25586187639656216,"accuracy":0.35502553053333724,"precision_macro":0.31443847752492743,"recall_macro":0.4615384888903963,"mcc":0.27013327061219017},
]

# -----------------------------
# 2) Table 1: Main comparison
# -----------------------------
table1 = pd.DataFrame([
    {"Method":"FP32 Baseline","Quantization Strategy":"No quantization", **fp32},
    {"Method":"Naive INT8","Quantization Strategy":"Dynamic quantization (all Linear)", **int8_naive},
    {"Method":"CAQ-Static (No Exclusion)","Quantization Strategy":"Static PTQ + class-balanced calibration (Percentile, per_class=2048)", **caq_static_no_excl},
    {"Method":"CAQ-Final (Proposed)","Quantization Strategy":"Static PTQ + class-balanced calibration + node exclusion (node_Gemm_0)", **caq_final},
])

# -----------------------------
# 3) Table 2: Calibration ablation
# -----------------------------
table2 = pd.DataFrame(calib_rows).sort_values(["method","per_class"], ascending=[True,True])

# -----------------------------
# 4) Table 3: Incremental gains
# -----------------------------
steps = [
    ("Naive Dynamic INT8", int8_naive["f1_macro"]),
    ("CAQ-Static (Balanced, no exclusion)", caq_static_no_excl["f1_macro"]),
    ("CAQ-Final (Balanced + exclusion)", caq_final["f1_macro"])
]
table3 = pd.DataFrame(steps, columns=["Step","F1_Macro"])
table3["Delta_vs_Previous"] = table3["F1_Macro"].diff()
table3["Delta_vs_Naive_INT8"] = table3["F1_Macro"] - int8_naive["f1_macro"]

# -----------------------------
# 5) Table 4: Compression-performance tradeoff
# -----------------------------
table4 = pd.DataFrame([
    {"Model":"FP32", **fp32},
    {"Model":"Naive INT8", **int8_naive},
    {"Model":"CAQ-Static (No Exclusion)", **caq_static_no_excl},
    {"Model":"CAQ-Final (Proposed)", **caq_final},
])
table4["Compression_vs_FP32_x"] = fp32["size_kb"] / table4["size_kb"]
table4["F1_Retention_vs_FP32_%"] = (table4["f1_macro"] / fp32["f1_macro"]) * 100

# -----------------------------
# 6) Table 5: Final summary
# -----------------------------
table5 = pd.DataFrame([
    {"Metric":"Best Method","Value":"CAQ-Final (Percentile static PTQ + class-balanced calibration + exclude node_Gemm_0)"},
    {"Metric":"F1-Macro","Value":caq_final["f1_macro"]},
    {"Metric":"Accuracy","Value":caq_final["accuracy"]},
    {"Metric":"Precision-Macro","Value":caq_final["precision_macro"]},
    {"Metric":"Recall-Macro","Value":caq_final["recall_macro"]},
    {"Metric":"MCC","Value":caq_final["mcc"]},
    {"Metric":"Model Size (KB)","Value":caq_final["size_kb"]},
    {"Metric":"Compression vs FP32 (x)","Value":fp32["size_kb"]/caq_final["size_kb"]},
    {"Metric":"F1 Gap to FP32","Value":fp32["f1_macro"]-caq_final["f1_macro"]},
])

# -----------------------------
# 7) Pretty print
# -----------------------------
pd.set_option("display.max_colwidth", 120)

print("\n=== TABLE 1: MAIN QUANTIZATION COMPARISON ===")
display(table1.round(4))

print("\n=== TABLE 2: CALIBRATION METHOD ABLATION ===")
display(table2.round(4))

print("\n=== TABLE 3: INCREMENTAL CAQ GAINS ===")
display(table3.round(4))

print("\n=== TABLE 4: COMPRESSION-PERFORMANCE TRADEOFF ===")
display(table4.round(4))

print("\n=== TABLE 5: FINAL PROPOSED METHOD SUMMARY ===")
display(table5)

# -----------------------------
# 8) Save CSVs for paper
# -----------------------------
table1.to_csv("paper_results/table1_main_quant_comparison.csv", index=False)
table2.to_csv("paper_results/table2_calibration_ablation.csv", index=False)
table3.to_csv("paper_results/table3_incremental_gains.csv", index=False)
table4.to_csv("paper_results/table4_tradeoff.csv", index=False)
table5.to_csv("paper_results/table5_final_summary.csv", index=False)

print("\n✅ Saved:")
print("paper_results/table1_main_quant_comparison.csv")
print("paper_results/table2_calibration_ablation.csv")
print("paper_results/table3_incremental_gains.csv")
print("paper_results/table4_tradeoff.csv")
print("paper_results/table5_final_summary.csv")